# SCF Phase 2 shard 29

Sweeps: power. Jobs: 5. Projected: 4.7 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 3, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.4, \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"f1e8a8f0f880\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/f1e8a8f0f880.parquet\", \"means_path\": \"data/sim/power/means/f1e8a8f0f880.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 3, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.8, \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"4b7578d5abfb\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/4b7578d5abfb.parquet\", \"means_path\": \"data/sim/power/means/4b7578d5abfb.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 3, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.6, \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"032b33bf6244\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/032b33bf6244.parquet\", \"means_path\": \"data/sim/power/means/032b33bf6244.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 3, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 3.2, \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"b19ba7fb2e34\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/b19ba7fb2e34.parquet\", \"means_path\": \"data/sim/power/means/b19ba7fb2e34.npz\", \"_rank\": 4, \"_sweep\": \"power\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.15, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"56e0e86c1555\", \"mode\": \"power\", \"sweep\": \"power\", \"reps\": 300, \"raw_path\": \"data/sim/power/raw/56e0e86c1555.parquet\", \"means_path\": \"data/sim/power/means/56e0e86c1555.npz\", \"_rank\": 4, \"_sweep\": \"power\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 29, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(29), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)